# Decision-Centric Fraud Detection — Thesis Pipeline

This notebook is the working implementation pipeline for the thesis.

The current stage focuses on:

1. reproducible project setup,
2. data loading and validation,
3. initial data quality checks,
4. exploratory data analysis,
5. a clean structure that leaves space for the next stages: predictive model, calibration, decision strategies, sequential simulation and monitoring.

> Important: this notebook is intentionally structured as a pipeline, not as a loose set of exploratory cells.

## 0. Notebook Setup

Centralised imports, display settings, plotting defaults and global configuration.

In [ ]:
# Core libraries
from pathlib import Path
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Warning and plot settings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 1. Project Configuration

All paths and target columns are defined here.  
This makes the notebook easier to move between machines without rewriting cells.

In [ ]:
# ---------------------------------------------------------------------
# USER CONFIGURATION
# ---------------------------------------------------------------------

# Change this path only if the dataset is stored somewhere else.
DATA_PATH = Path(r"C:\Users\PATRI\OneDrive\Thesis\AIML Dataset.csv")

# Core column names
TARGET_COL = "isFraud"
SYSTEM_FLAG_COL = "isFlaggedFraud"
TIME_COL = "step"
AMOUNT_COL = "amount"
TRANSACTION_TYPE_COL = "type"

# Entity identifiers
ORIGIN_ID_COL = "nameOrig"
DESTINATION_ID_COL = "nameDest"

# Columns that must exist in the raw dataset
REQUIRED_COLUMNS = [
    TIME_COL,
    TRANSACTION_TYPE_COL,
    AMOUNT_COL,
    ORIGIN_ID_COL,
    "oldbalanceOrg",
    "newbalanceOrig",
    DESTINATION_ID_COL,
    "oldbalanceDest",
    "newbalanceDest",
    TARGET_COL,
    SYSTEM_FLAG_COL,
]

## 2. Data Loading

Load the raw transaction dataset and perform basic structural validation.

In [ ]:
def load_transactions(path: Path) -> pd.DataFrame:
    """Load transaction data from CSV."""
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at: {path}\n"
            "Update DATA_PATH in the Project Configuration section."
        )
    return pd.read_csv(path)


def validate_schema(data: pd.DataFrame, required_columns: list[str]) -> None:
    """Check whether all required columns are present."""
    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    print("Schema validation passed.")


df_raw = load_transactions(DATA_PATH)
validate_schema(df_raw, REQUIRED_COLUMNS)

print(f"Dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")
df_raw.head()

## 3. Initial Data Overview

Basic checks before any modelling decision.

In [ ]:
df_raw.info()

In [ ]:
def summarise_dataframe(data: pd.DataFrame) -> pd.DataFrame:
    """Return a compact overview of column types, missing values and uniqueness."""
    summary = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "missing_count": data.isna().sum(),
        "missing_rate": data.isna().mean(),
        "unique_values": data.nunique(dropna=False),
    })
    return summary.sort_values("missing_rate", ascending=False)

summarise_dataframe(df_raw)

In [ ]:
# Duplicate rows check
duplicate_rows = df_raw.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows:,}")

## 4. Target and Class Imbalance

Fraud detection is expected to be highly imbalanced, so accuracy will not be a suitable primary metric.

In [ ]:
def target_distribution(data: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """Return count and percentage distribution for a binary target."""
    counts = data[target_col].value_counts(dropna=False).rename("count")
    rates = data[target_col].value_counts(normalize=True, dropna=False).rename("rate")
    return pd.concat([counts, rates], axis=1)

fraud_distribution = target_distribution(df_raw, TARGET_COL)
fraud_distribution

In [ ]:
fraud_rate = df_raw[TARGET_COL].mean()
print(f"Fraud rate: {fraud_rate:.4%}")

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_raw, x=TARGET_COL)
plt.title("Target Distribution: Fraud vs Non-Fraud")
plt.xlabel("isFraud")
plt.ylabel("Transaction Count")
plt.show()

## 5. System Flag Inspection

`isFlaggedFraud` is a rule/system-generated flag.  
It can be useful for analysis, but it should not be used as a predictive feature because it may leak operational information.

In [ ]:
flag_distribution = target_distribution(df_raw, SYSTEM_FLAG_COL)
flag_distribution

In [ ]:
pd.crosstab(
    df_raw[SYSTEM_FLAG_COL],
    df_raw[TARGET_COL],
    rownames=[SYSTEM_FLAG_COL],
    colnames=[TARGET_COL],
    margins=True,
)

## 6. Exploratory Data Analysis

This section explores the most important raw patterns before feature engineering.

### 6.1 Transaction Types

In [ ]:
type_counts = df_raw[TRANSACTION_TYPE_COL].value_counts()
type_counts

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(
    data=df_raw,
    x=TRANSACTION_TYPE_COL,
    order=type_counts.index,
)
plt.title("Transaction Count by Type")
plt.xlabel("Transaction Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
fraud_rate_by_type = (
    df_raw.groupby(TRANSACTION_TYPE_COL)[TARGET_COL]
    .agg(fraud_rate="mean", fraud_count="sum", total_count="count")
    .sort_values("fraud_rate", ascending=False)
)

fraud_rate_by_type

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=fraud_rate_by_type.reset_index(),
    x=TRANSACTION_TYPE_COL,
    y="fraud_rate",
)
plt.title("Fraud Rate by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate")
plt.xticks(rotation=45)
plt.show()

### 6.2 Transaction Amount

In [ ]:
df_raw[AMOUNT_COL].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(np.log1p(df_raw[AMOUNT_COL]), bins=100, kde=True)
plt.title("Transaction Amount Distribution — Log Scale")
plt.xlabel("log1p(amount)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_raw, x=TARGET_COL, y=np.log1p(df_raw[AMOUNT_COL]))
plt.title("Transaction Amount by Fraud Label — Log Scale")
plt.xlabel("isFraud")
plt.ylabel("log1p(amount)")
plt.show()

### 6.3 Time Variable

The `step` column will later be used for temporal splitting and sequential simulation.

In [ ]:
df_raw[TIME_COL].describe().to_frame()

In [ ]:
transactions_by_step = df_raw.groupby(TIME_COL).size().rename("transaction_count")
frauds_by_step = df_raw.groupby(TIME_COL)[TARGET_COL].sum().rename("fraud_count")

step_summary = pd.concat([transactions_by_step, frauds_by_step], axis=1).fillna(0)
step_summary["fraud_rate"] = step_summary["fraud_count"] / step_summary["transaction_count"]
step_summary.head()

In [ ]:
plt.figure(figsize=(10, 4))
step_summary["transaction_count"].plot()
plt.title("Transaction Volume Over Time")
plt.xlabel("Step")
plt.ylabel("Transaction Count")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
step_summary["fraud_count"].plot()
plt.title("Fraud Count Over Time")
plt.xlabel("Step")
plt.ylabel("Fraud Count")
plt.show()

## 7. Clean Working Copy

Create a separate working dataframe. Raw data stays untouched in `df_raw`.

In [ ]:
df = df_raw.copy()

# Sort by time to prepare for temporal split and sequential simulation.
df = df.sort_values(TIME_COL).reset_index(drop=True)

print(f"Working dataframe shape: {df.shape}")
df.head()

## 8. Feature Engineering Placeholder

This is the next major section. For now, we define the structure without committing to final modelling choices.

In [ ]:
def add_basic_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Add leakage-safe transaction-level features.

    These features use only information available at transaction time.
    More advanced entity-history features can be added later, but must be built
    carefully to avoid using future information.
    """
    data = data.copy()

    # Amount transformation
    data["log_amount"] = np.log1p(data[AMOUNT_COL])

    # Balance consistency features
    data["origin_balance_error"] = (
        data["oldbalanceOrg"] - data[AMOUNT_COL] - data["newbalanceOrig"]
    )
    data["destination_balance_error"] = (
        data["oldbalanceDest"] + data[AMOUNT_COL] - data["newbalanceDest"]
    )

    # Absolute versions often work better for tree models and anomaly-like patterns
    data["abs_origin_balance_error"] = data["origin_balance_error"].abs()
    data["abs_destination_balance_error"] = data["destination_balance_error"].abs()

    return data


df_features = add_basic_features(df)
df_features.head()

## 9. Modelling Dataset Placeholder

`isFlaggedFraud` is deliberately excluded from model features.
Entity IDs are also excluded in the first baseline model; they can later be used to build leakage-safe historical features.

In [ ]:
EXCLUDED_FROM_MODEL = [
    TARGET_COL,
    SYSTEM_FLAG_COL,
    ORIGIN_ID_COL,
    DESTINATION_ID_COL,
]

candidate_features = [
    col for col in df_features.columns
    if col not in EXCLUDED_FROM_MODEL
]

print(f"Candidate feature count: {len(candidate_features)}")
candidate_features

## 10. Temporal Split Placeholder

The thesis requires realistic evaluation. Random split should be avoided because it can overestimate performance in time-dependent fraud data.

In [ ]:
def temporal_train_valid_test_split(
    data: pd.DataFrame,
    time_col: str,
    train_size: float = 0.60,
    valid_size: float = 0.20,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split data chronologically into train, validation and test sets."""
    if train_size <= 0 or valid_size <= 0 or train_size + valid_size >= 1:
        raise ValueError("train_size and valid_size must be positive and sum to less than 1.")

    data = data.sort_values(time_col).reset_index(drop=True)
    n_rows = len(data)

    train_end = int(n_rows * train_size)
    valid_end = int(n_rows * (train_size + valid_size))

    train = data.iloc[:train_end].copy()
    valid = data.iloc[train_end:valid_end].copy()
    test = data.iloc[valid_end:].copy()

    return train, valid, test


train_df, valid_df, test_df = temporal_train_valid_test_split(df_features, TIME_COL)

print(f"Train:      {train_df.shape}")
print(f"Validation: {valid_df.shape}")
print(f"Test:       {test_df.shape}")

In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(valid_df), len(test_df)],
    "min_step": [train_df[TIME_COL].min(), valid_df[TIME_COL].min(), test_df[TIME_COL].min()],
    "max_step": [train_df[TIME_COL].max(), valid_df[TIME_COL].max(), test_df[TIME_COL].max()],
    "fraud_rate": [train_df[TARGET_COL].mean(), valid_df[TARGET_COL].mean(), test_df[TARGET_COL].mean()],
    "fraud_count": [train_df[TARGET_COL].sum(), valid_df[TARGET_COL].sum(), test_df[TARGET_COL].sum()],
})

split_summary

## 11. Next Pipeline Slots

The following sections are intentionally left as structured placeholders.
They define where the thesis implementation should continue.

### 11.1 Predictive Model

This section turns the cleaned transaction table into a reproducible modelling pipeline.  
The goal is **not** to compare many classifiers. The classifier is only the scoring layer.  
The thesis comparison happens later at the **decision layer**.

In [ ]:
# ---------------------------------------------------------------------
# Predictive modelling setup
# ---------------------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Keep the first baseline simple and explainable.
# Do not use isFlaggedFraud, nameOrig, or nameDest as raw predictive inputs.
MODEL_FEATURES = candidate_features.copy()

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df[TARGET_COL].astype(int).copy()

X_valid = valid_df[MODEL_FEATURES].copy()
y_valid = valid_df[TARGET_COL].astype(int).copy()

X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df[TARGET_COL].astype(int).copy()

categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = [col for col in MODEL_FEATURES if col not in categorical_features]

print(f"Numeric features: {len(numeric_features)}")
print(numeric_features)
print(f"\nCategorical features: {len(categorical_features)}")
print(categorical_features)


In [ ]:
def make_preprocessing_pipeline(numeric_features: list[str], categorical_features: list[str]) -> ColumnTransformer:
    """Create preprocessing pipeline for numeric and categorical transaction features."""
    numeric_transformer = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ],
        remainder="drop",
    )

    return preprocessor


def make_baseline_model() -> Pipeline:
    """
    Build the baseline fraud scoring model.

    Logistic Regression is deliberately used as the first baseline because:
    - it is transparent,
    - it gives probability scores,
    - it supports class weighting for rare fraud cases,
    - it is easier to defend methodologically than jumping straight to a black-box model.
    """
    preprocessor = make_preprocessing_pipeline(numeric_features, categorical_features)

    classifier = LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=500,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("classifier", classifier),
        ]
    )

    return model

baseline_model = make_baseline_model()
baseline_model


In [ ]:
# ---------------------------------------------------------------------
# Optional speed control
# ---------------------------------------------------------------------
# Large fraud datasets can be slow with full training during development.
# Keep TRAIN_SAMPLE_FRACTION = 1.0 for final thesis results.

TRAIN_SAMPLE_FRACTION = 1.0

if TRAIN_SAMPLE_FRACTION < 1.0:
    train_sample = train_df.groupby(TARGET_COL, group_keys=False).apply(
        lambda part: part.sample(
            frac=TRAIN_SAMPLE_FRACTION,
            random_state=RANDOM_STATE,
        )
    )
    X_train_fit = train_sample[MODEL_FEATURES].copy()
    y_train_fit = train_sample[TARGET_COL].astype(int).copy()
else:
    X_train_fit = X_train
    y_train_fit = y_train

print(f"Training rows used: {len(X_train_fit):,}")
print(y_train_fit.value_counts(normalize=True).rename("class_rate"))


In [ ]:
# Train baseline scoring model
baseline_model.fit(X_train_fit, y_train_fit)

valid_scores_uncalibrated = baseline_model.predict_proba(X_valid)[:, 1]
test_scores_uncalibrated = baseline_model.predict_proba(X_test)[:, 1]

print("Baseline scoring model trained.")


In [ ]:
def evaluate_scores(y_true: pd.Series, scores: np.ndarray, threshold: float = 0.5) -> dict:
    """Evaluate fraud probability scores with ranking and threshold-based metrics."""
    predictions = (scores >= threshold).astype(int)

    return {
        "threshold": threshold,
        "PR_AUC": average_precision_score(y_true, scores),
        "ROC_AUC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "alerts": int(predictions.sum()),
    }

score_evaluation = pd.DataFrame([
    {"split": "validation", **evaluate_scores(y_valid, valid_scores_uncalibrated)},
    {"split": "test", **evaluate_scores(y_test, test_scores_uncalibrated)},
])

score_evaluation


### 11.2 Probability Calibration

The decision layer uses fraud probabilities. That means raw classifier scores are not enough.  
Calibration checks whether a score of 0.80 behaves like an actual 80% fraud risk.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss


def make_prefit_calibrator(fitted_model, method: str = "sigmoid"):
    """
    Create a calibrator fitted on validation data.

    Newer sklearn versions use FrozenEstimator. Older versions use cv='prefit'.
    This helper supports both, so the notebook is less fragile across environments.
    """
    try:
        from sklearn.frozen import FrozenEstimator
        return CalibratedClassifierCV(FrozenEstimator(fitted_model), method=method)
    except Exception:
        return CalibratedClassifierCV(fitted_model, method=method, cv="prefit")

calibrator = make_prefit_calibrator(baseline_model, method="sigmoid")
calibrator.fit(X_valid, y_valid)

valid_scores_calibrated = calibrator.predict_proba(X_valid)[:, 1]
test_scores_calibrated = calibrator.predict_proba(X_test)[:, 1]

calibration_summary = pd.DataFrame([
    {
        "split": "validation",
        "score_type": "uncalibrated",
        "PR_AUC": average_precision_score(y_valid, valid_scores_uncalibrated),
        "ROC_AUC": roc_auc_score(y_valid, valid_scores_uncalibrated),
        "Brier": brier_score_loss(y_valid, valid_scores_uncalibrated),
    },
    {
        "split": "validation",
        "score_type": "calibrated",
        "PR_AUC": average_precision_score(y_valid, valid_scores_calibrated),
        "ROC_AUC": roc_auc_score(y_valid, valid_scores_calibrated),
        "Brier": brier_score_loss(y_valid, valid_scores_calibrated),
    },
    {
        "split": "test",
        "score_type": "uncalibrated",
        "PR_AUC": average_precision_score(y_test, test_scores_uncalibrated),
        "ROC_AUC": roc_auc_score(y_test, test_scores_uncalibrated),
        "Brier": brier_score_loss(y_test, test_scores_uncalibrated),
    },
    {
        "split": "test",
        "score_type": "calibrated",
        "PR_AUC": average_precision_score(y_test, test_scores_calibrated),
        "ROC_AUC": roc_auc_score(y_test, test_scores_calibrated),
        "Brier": brier_score_loss(y_test, test_scores_calibrated),
    },
])

calibration_summary


In [ ]:
def plot_calibration_curve(y_true, uncalibrated_scores, calibrated_scores, n_bins: int = 10):
    """Plot calibration curve before and after calibration."""
    prob_true_uncal, prob_pred_uncal = calibration_curve(
        y_true, uncalibrated_scores, n_bins=n_bins, strategy="quantile"
    )
    prob_true_cal, prob_pred_cal = calibration_curve(
        y_true, calibrated_scores, n_bins=n_bins, strategy="quantile"
    )

    plt.figure(figsize=(6, 6))
    plt.plot(prob_pred_uncal, prob_true_uncal, marker="o", label="Uncalibrated")
    plt.plot(prob_pred_cal, prob_true_cal, marker="o", label="Calibrated")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.title("Calibration Curve — Validation Set")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed fraud rate")
    plt.legend()
    plt.show()

plot_calibration_curve(y_valid, valid_scores_uncalibrated, valid_scores_calibrated)


In [ ]:
# Use calibrated probabilities as the main fraud risk score for the decision layer.
valid_decision_df = valid_df.copy()
valid_decision_df["fraud_score"] = valid_scores_calibrated

# The test set remains the final holdout for reporting.
test_decision_df = test_df.copy()
test_decision_df["fraud_score"] = test_scores_calibrated

print(valid_decision_df[[TIME_COL, AMOUNT_COL, TARGET_COL, "fraud_score"]].head())


### 11.3 Decision Strategies

This is the core thesis layer.  
The same predictive scores are converted into decisions using four strategies:

1. **Static threshold** baseline  
2. **Top-k adaptive selection** under alert capacity  
3. **Cost-aware ranking**  
4. **Full decision system** with adaptive selection, cost-awareness and suppression

In [ ]:
# ---------------------------------------------------------------------
# Decision strategy parameters
# ---------------------------------------------------------------------

STATIC_THRESHOLD = 0.50
ALERT_BUDGET_PER_STEP = 50
INVESTIGATION_COST = 10.0
SUPPRESSION_WINDOW = 24  # PaySim step is usually one hour; 24 approximates one day.
ENTITY_COL_FOR_SUPPRESSION = ORIGIN_ID_COL

print("Decision parameters")
print(f"Static threshold: {STATIC_THRESHOLD}")
print(f"Alert budget per step: {ALERT_BUDGET_PER_STEP}")
print(f"Investigation cost: {INVESTIGATION_COST}")
print(f"Suppression window: {SUPPRESSION_WINDOW}")
print(f"Suppression entity column: {ENTITY_COL_FOR_SUPPRESSION}")


In [ ]:
def apply_static_threshold(
    data: pd.DataFrame,
    score_col: str,
    threshold: float,
    decision_col: str = "alert_static_threshold",
) -> pd.DataFrame:
    """Flag every transaction whose fraud score exceeds a fixed threshold."""
    result = data.copy()
    result[decision_col] = (result[score_col] >= threshold).astype(int)
    return result


def apply_top_k_selection(
    data: pd.DataFrame,
    score_col: str,
    k: int,
    window_col: str,
    decision_col: str = "alert_top_k",
) -> pd.DataFrame:
    """Select the top-k highest risk transactions inside each time window."""
    result = data.copy()
    result[decision_col] = 0

    selected_indices = (
        result
        .sort_values([window_col, score_col], ascending=[True, False])
        .groupby(window_col, group_keys=False)
        .head(k)
        .index
    )

    result.loc[selected_indices, decision_col] = 1
    return result


def add_expected_benefit_score(
    data: pd.DataFrame,
    probability_col: str,
    amount_col: str,
    investigation_cost: float,
    benefit_col: str = "expected_benefit",
) -> pd.DataFrame:
    """
    Estimate expected benefit from investigating a transaction.

    Simple decision logic:
    expected_benefit = P(fraud) * transaction_amount - investigation_cost

    Positive value means the expected avoided fraud loss is greater than the review cost.
    """
    result = data.copy()
    result[benefit_col] = result[probability_col] * result[amount_col] - investigation_cost
    return result


def apply_cost_aware_ranking(
    data: pd.DataFrame,
    probability_col: str,
    amount_col: str,
    investigation_cost: float,
    k: int | None = None,
    window_col: str | None = None,
    decision_col: str = "alert_cost_aware",
) -> pd.DataFrame:
    """
    Flag transactions using expected benefit.

    If k and window_col are provided, select the top-k positive-benefit transactions
    per window. Otherwise, flag all positive-benefit transactions.
    """
    result = add_expected_benefit_score(
        data=data,
        probability_col=probability_col,
        amount_col=amount_col,
        investigation_cost=investigation_cost,
    )
    result[decision_col] = 0

    eligible = result[result["expected_benefit"] > 0].copy()

    if k is None or window_col is None:
        result.loc[eligible.index, decision_col] = 1
    else:
        selected_indices = (
            eligible
            .sort_values([window_col, "expected_benefit"], ascending=[True, False])
            .groupby(window_col, group_keys=False)
            .head(k)
            .index
        )
        result.loc[selected_indices, decision_col] = 1

    return result


def apply_entity_suppression(
    data: pd.DataFrame,
    decision_col: str,
    entity_col: str,
    time_col: str,
    suppression_window: int,
    suppressed_col: str = "suppressed_alert",
    final_decision_col: str = "alert_after_suppression",
) -> pd.DataFrame:
    """
    Suppress repeated alerts for the same entity within a time window.

    The first alert is kept. Later alerts for the same entity are suppressed if they occur
    within suppression_window time units from the last kept alert.
    """
    result = data.sort_values(time_col).copy()
    result[suppressed_col] = 0
    result[final_decision_col] = 0

    last_kept_alert_time = {}

    for idx, row in result.iterrows():
        if row[decision_col] != 1:
            continue

        entity = row[entity_col]
        current_time = row[time_col]
        last_time = last_kept_alert_time.get(entity)

        if last_time is not None and (current_time - last_time) <= suppression_window:
            result.at[idx, suppressed_col] = 1
        else:
            result.at[idx, final_decision_col] = 1
            last_kept_alert_time[entity] = current_time

    return result.sort_index()


def apply_full_decision_system(
    data: pd.DataFrame,
    probability_col: str,
    amount_col: str,
    investigation_cost: float,
    k: int,
    window_col: str,
    entity_col: str,
    time_col: str,
    suppression_window: int,
) -> pd.DataFrame:
    """Apply cost-aware top-k selection and then suppress repeated entity alerts."""
    result = apply_cost_aware_ranking(
        data=data,
        probability_col=probability_col,
        amount_col=amount_col,
        investigation_cost=investigation_cost,
        k=k,
        window_col=window_col,
        decision_col="alert_full_candidate",
    )

    result = apply_entity_suppression(
        data=result,
        decision_col="alert_full_candidate",
        entity_col=entity_col,
        time_col=time_col,
        suppression_window=suppression_window,
        suppressed_col="suppressed_alert",
        final_decision_col="alert_full_system",
    )

    return result


In [ ]:
# Apply all decision strategies on validation and test sets.
# Validation can be used to tune thresholds, budgets, cost assumptions and suppression windows.
# Test is used for final reporting only.

valid_decisions = valid_decision_df.copy()
valid_decisions = apply_static_threshold(valid_decisions, "fraud_score", STATIC_THRESHOLD)
valid_decisions = apply_top_k_selection(valid_decisions, "fraud_score", ALERT_BUDGET_PER_STEP, TIME_COL)
valid_decisions = apply_cost_aware_ranking(
    valid_decisions,
    probability_col="fraud_score",
    amount_col=AMOUNT_COL,
    investigation_cost=INVESTIGATION_COST,
    k=ALERT_BUDGET_PER_STEP,
    window_col=TIME_COL,
)
valid_decisions = apply_full_decision_system(
    valid_decisions,
    probability_col="fraud_score",
    amount_col=AMOUNT_COL,
    investigation_cost=INVESTIGATION_COST,
    k=ALERT_BUDGET_PER_STEP,
    window_col=TIME_COL,
    entity_col=ENTITY_COL_FOR_SUPPRESSION,
    time_col=TIME_COL,
    suppression_window=SUPPRESSION_WINDOW,
)

test_decisions = test_decision_df.copy()
test_decisions = apply_static_threshold(test_decisions, "fraud_score", STATIC_THRESHOLD)
test_decisions = apply_top_k_selection(test_decisions, "fraud_score", ALERT_BUDGET_PER_STEP, TIME_COL)
test_decisions = apply_cost_aware_ranking(
    test_decisions,
    probability_col="fraud_score",
    amount_col=AMOUNT_COL,
    investigation_cost=INVESTIGATION_COST,
    k=ALERT_BUDGET_PER_STEP,
    window_col=TIME_COL,
)
test_decisions = apply_full_decision_system(
    test_decisions,
    probability_col="fraud_score",
    amount_col=AMOUNT_COL,
    investigation_cost=INVESTIGATION_COST,
    k=ALERT_BUDGET_PER_STEP,
    window_col=TIME_COL,
    entity_col=ENTITY_COL_FOR_SUPPRESSION,
    time_col=TIME_COL,
    suppression_window=SUPPRESSION_WINDOW,
)

valid_decisions[[TARGET_COL, AMOUNT_COL, "fraud_score", "alert_static_threshold", "alert_top_k", "alert_cost_aware", "alert_full_system"]].head()


### 11.4 Operational and Cost-Based Evaluation

This section evaluates the thesis question directly: not only whether fraud is predicted, but whether the system creates a useful and manageable set of alerts.

In [ ]:
def evaluate_decision_strategy(
    data: pd.DataFrame,
    y_true_col: str,
    decision_col: str,
    amount_col: str,
    investigation_cost: float,
    strategy_name: str,
    suppressed_col: str | None = None,
) -> dict:
    """Evaluate a decision strategy with predictive, operational and cost metrics."""
    y_true = data[y_true_col].astype(int)
    alerts = data[decision_col].astype(int)

    tp_mask = (alerts == 1) & (y_true == 1)
    fp_mask = (alerts == 1) & (y_true == 0)
    fn_mask = (alerts == 0) & (y_true == 1)
    tn_mask = (alerts == 0) & (y_true == 0)

    alert_count = int(alerts.sum())
    fraud_count = int(y_true.sum())
    frauds_caught = int(tp_mask.sum())
    false_positives = int(fp_mask.sum())
    missed_frauds = int(fn_mask.sum())

    investigation_total_cost = alert_count * investigation_cost
    missed_fraud_loss = float(data.loc[fn_mask, amount_col].sum())
    caught_fraud_amount = float(data.loc[tp_mask, amount_col].sum())
    total_operational_cost = investigation_total_cost + missed_fraud_loss

    suppressed_count = int(data[suppressed_col].sum()) if suppressed_col and suppressed_col in data.columns else 0
    candidate_alerts = alert_count + suppressed_count
    suppression_rate = suppressed_count / candidate_alerts if candidate_alerts > 0 else 0

    return {
        "strategy": strategy_name,
        "transactions": len(data),
        "frauds_total": fraud_count,
        "alerts": alert_count,
        "frauds_caught": frauds_caught,
        "missed_frauds": missed_frauds,
        "false_positives": false_positives,
        "precision_alerts": frauds_caught / alert_count if alert_count > 0 else 0,
        "recall_fraud": frauds_caught / fraud_count if fraud_count > 0 else 0,
        "false_positive_load": false_positives,
        "alert_rate": alert_count / len(data) if len(data) > 0 else 0,
        "suppressed_alerts": suppressed_count,
        "suppression_rate": suppression_rate,
        "investigation_cost": investigation_total_cost,
        "missed_fraud_loss": missed_fraud_loss,
        "caught_fraud_amount": caught_fraud_amount,
        "total_operational_cost": total_operational_cost,
        "cost_per_alert": total_operational_cost / alert_count if alert_count > 0 else np.nan,
    }


def compare_decision_strategies(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Compare all thesis decision strategies on one split."""
    rows = [
        evaluate_decision_strategy(
            data, TARGET_COL, "alert_static_threshold", AMOUNT_COL, INVESTIGATION_COST, "static_threshold"
        ),
        evaluate_decision_strategy(
            data, TARGET_COL, "alert_top_k", AMOUNT_COL, INVESTIGATION_COST, "top_k_adaptive"
        ),
        evaluate_decision_strategy(
            data, TARGET_COL, "alert_cost_aware", AMOUNT_COL, INVESTIGATION_COST, "cost_aware_ranking"
        ),
        evaluate_decision_strategy(
            data, TARGET_COL, "alert_full_system", AMOUNT_COL, INVESTIGATION_COST, "full_system_with_suppression", suppressed_col="suppressed_alert"
        ),
    ]

    comparison = pd.DataFrame(rows)
    comparison.insert(0, "split", split_name)
    return comparison

valid_strategy_comparison = compare_decision_strategies(valid_decisions, "validation")
test_strategy_comparison = compare_decision_strategies(test_decisions, "test")

strategy_comparison = pd.concat([valid_strategy_comparison, test_strategy_comparison], ignore_index=True)
strategy_comparison


In [ ]:
# Compact thesis-facing table
report_cols = [
    "split",
    "strategy",
    "alerts",
    "frauds_caught",
    "precision_alerts",
    "recall_fraud",
    "false_positive_load",
    "suppression_rate",
    "missed_fraud_loss",
    "investigation_cost",
    "total_operational_cost",
]

strategy_comparison[report_cols].sort_values(["split", "total_operational_cost"])


In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(
    data=test_strategy_comparison,
    x="strategy",
    y="total_operational_cost",
)
plt.title("Decision Strategy Comparison — Total Operational Cost on Test Set")
plt.xlabel("Decision Strategy")
plt.ylabel("Total Operational Cost")
plt.xticks(rotation=30, ha="right")
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(
    data=test_strategy_comparison,
    x="strategy",
    y="alerts",
)
plt.title("Decision Strategy Comparison — Alert Volume on Test Set")
plt.xlabel("Decision Strategy")
plt.ylabel("Alert Count")
plt.xticks(rotation=30, ha="right")
plt.show()


### 11.5 Sequential Simulation and Monitoring

Batch evaluation answers: “How did the strategy perform overall?”  
Sequential simulation answers: “How would this behave if transactions arrived over time?”

This is where the thesis becomes stronger than a standard fraud classification project.

In [ ]:
def run_sequential_simulation(
    data: pd.DataFrame,
    score_col: str,
    time_col: str,
    amount_col: str,
    target_col: str,
    strategy: str,
    k: int,
    investigation_cost: float,
    threshold: float = 0.5,
    entity_col: str | None = None,
    suppression_window: int | None = None,
) -> pd.DataFrame:
    """
    Process transactions window-by-window in chronological order.

    Supported strategies:
    - static_threshold
    - top_k_adaptive
    - cost_aware_ranking
    - full_system_with_suppression
    """
    ordered = data.sort_values(time_col).copy()
    outputs = []

    last_kept_alert_time = {}

    for window_value, window_data in ordered.groupby(time_col, sort=True):
        window_result = window_data.copy()
        window_result["sequential_alert"] = 0
        window_result["sequential_suppressed"] = 0

        if strategy == "static_threshold":
            selected_idx = window_result[window_result[score_col] >= threshold].index

        elif strategy == "top_k_adaptive":
            selected_idx = window_result.sort_values(score_col, ascending=False).head(k).index

        elif strategy == "cost_aware_ranking":
            window_result["expected_benefit"] = (
                window_result[score_col] * window_result[amount_col] - investigation_cost
            )
            eligible = window_result[window_result["expected_benefit"] > 0]
            selected_idx = eligible.sort_values("expected_benefit", ascending=False).head(k).index

        elif strategy == "full_system_with_suppression":
            if entity_col is None or suppression_window is None:
                raise ValueError("entity_col and suppression_window are required for full_system_with_suppression")

            window_result["expected_benefit"] = (
                window_result[score_col] * window_result[amount_col] - investigation_cost
            )
            candidates = (
                window_result[window_result["expected_benefit"] > 0]
                .sort_values("expected_benefit", ascending=False)
                .head(k)
            )

            selected = []
            for idx, row in candidates.sort_values(time_col).iterrows():
                entity = row[entity_col]
                current_time = row[time_col]
                last_time = last_kept_alert_time.get(entity)

                if last_time is not None and (current_time - last_time) <= suppression_window:
                    window_result.at[idx, "sequential_suppressed"] = 1
                else:
                    selected.append(idx)
                    last_kept_alert_time[entity] = current_time

            selected_idx = selected

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        window_result.loc[selected_idx, "sequential_alert"] = 1
        outputs.append(window_result)

    return pd.concat(outputs).sort_index()


def build_monitoring_table(
    sequential_data: pd.DataFrame,
    time_col: str,
    target_col: str,
    alert_col: str,
    amount_col: str,
    investigation_cost: float,
    monitoring_window: int = 24,
) -> pd.DataFrame:
    """Aggregate sequential performance into monitoring windows."""
    monitored = sequential_data.copy()
    monitored["monitoring_window"] = (monitored[time_col] // monitoring_window).astype(int)

    rows = []
    for window_id, part in monitored.groupby("monitoring_window", sort=True):
        metrics = evaluate_decision_strategy(
            data=part,
            y_true_col=target_col,
            decision_col=alert_col,
            amount_col=amount_col,
            investigation_cost=investigation_cost,
            strategy_name=f"window_{window_id}",
            suppressed_col="sequential_suppressed" if "sequential_suppressed" in part.columns else None,
        )
        metrics["monitoring_window"] = window_id
        metrics["min_step"] = part[time_col].min()
        metrics["max_step"] = part[time_col].max()
        rows.append(metrics)

    return pd.DataFrame(rows)


In [ ]:
# Run sequential simulation for all strategies on the final test set.
sequential_results = {}
monitoring_results = {}

for strategy in [
    "static_threshold",
    "top_k_adaptive",
    "cost_aware_ranking",
    "full_system_with_suppression",
]:
    simulated = run_sequential_simulation(
        data=test_decision_df,
        score_col="fraud_score",
        time_col=TIME_COL,
        amount_col=AMOUNT_COL,
        target_col=TARGET_COL,
        strategy=strategy,
        k=ALERT_BUDGET_PER_STEP,
        investigation_cost=INVESTIGATION_COST,
        threshold=STATIC_THRESHOLD,
        entity_col=ENTITY_COL_FOR_SUPPRESSION,
        suppression_window=SUPPRESSION_WINDOW,
    )

    sequential_results[strategy] = simulated
    monitoring_results[strategy] = build_monitoring_table(
        sequential_data=simulated,
        time_col=TIME_COL,
        target_col=TARGET_COL,
        alert_col="sequential_alert",
        amount_col=AMOUNT_COL,
        investigation_cost=INVESTIGATION_COST,
        monitoring_window=24,
    )

sequential_summary = pd.DataFrame([
    evaluate_decision_strategy(
        data=simulated,
        y_true_col=TARGET_COL,
        decision_col="sequential_alert",
        amount_col=AMOUNT_COL,
        investigation_cost=INVESTIGATION_COST,
        strategy_name=strategy,
        suppressed_col="sequential_suppressed",
    )
    for strategy, simulated in sequential_results.items()
])

sequential_summary


In [ ]:
# Compare batch-style test decisions with sequential simulation.
batch_vs_sequential = pd.concat([
    test_strategy_comparison.assign(evaluation_mode="batch"),
    sequential_summary.assign(split="test", evaluation_mode="sequential"),
], ignore_index=True, sort=False)

batch_vs_sequential[[
    "evaluation_mode",
    "strategy",
    "alerts",
    "frauds_caught",
    "precision_alerts",
    "recall_fraud",
    "suppression_rate",
    "total_operational_cost",
]].sort_values(["strategy", "evaluation_mode"])


In [ ]:
# Monitoring example: full system behaviour over time.
full_monitoring = monitoring_results["full_system_with_suppression"]

plt.figure(figsize=(10, 4))
plt.plot(full_monitoring["monitoring_window"], full_monitoring["alerts"], marker="o")
plt.title("Monitoring — Alert Volume Over Time, Full Decision System")
plt.xlabel("Monitoring Window")
plt.ylabel("Alerts")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(full_monitoring["monitoring_window"], full_monitoring["precision_alerts"], marker="o")
plt.title("Monitoring — Alert Precision Over Time, Full Decision System")
plt.xlabel("Monitoring Window")
plt.ylabel("Alert Precision")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(full_monitoring["monitoring_window"], full_monitoring["total_operational_cost"], marker="o")
plt.title("Monitoring — Operational Cost Over Time, Full Decision System")
plt.xlabel("Monitoring Window")
plt.ylabel("Total Operational Cost")
plt.show()


### 11.6 Thesis Interpretation Checklist

Use this checklist after running the notebook:

- Does top-k reduce alert volume compared with static threshold?
- Does cost-aware ranking reduce total operational cost?
- Does suppression reduce repeated alerts without destroying recall?
- Do batch and sequential results differ?
- Are the final conclusions based on the decision layer, not just model metrics?

If the answer to these is visible in the tables above, the implementation finally matches the thesis direction.